# 命令行、日志与配置

学习目标：为小型脚本设计可检查的命令行输入、配置优先级与日志输出，并可靠地调用和约束子进程。

前置知识：函数、模块导入、列表与字典、异常处理、with、文本文件以及 JSON 和 TOML 的基本结构。

运行环境：Python 3.12。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例启动本地 Python 子进程，临时文件自动清理。

配套脚本：位于 [scripts/22-cli-logging-config/](scripts/22-cli-logging-config/)。

（1）[study\_cli.py](scripts/22-cli-logging-config/study_cli.py)：可直接运行的学习标签 CLI，演示参数解析、帮助信息与退出状态。

## 1 sys.argv 接收到什么

命令行接口（command-line interface，CLI）把启动参数交给程序。sys.argv 是字符串列表；运行脚本时第 0 项是脚本名称，后续项才是用户参数。用 Python 的 -c 执行一段代码时，第 0 项为 -c。

下面用 subprocess.run 启动本地 Python，等待结束并捕获输出。参数列表中的每一项对应一个参数，带空格的标题仍放在同一个字符串里；第 7 节继续说明退出检查与超时。

In [1]:
import json
import os
from pathlib import Path
import subprocess
import sys

# 仅改变子进程的编码和字节码选项，Notebook 本身的环境保持原状。
child_env = os.environ.copy()
child_env["PYTHONDONTWRITEBYTECODE"] = "1"
child_env["PYTHONIOENCODING"] = "utf-8"
probe = "import json, sys; print(json.dumps(sys.argv, ensure_ascii=False))"
# 列表中的每项是一个参数，带空格的标题仍作为完整字符串传入。
result = subprocess.run(
    [sys.executable, "-B", "-c", probe, "Python 学习", "--count", "2"],
    shell=False,
    capture_output=True,
    encoding="utf-8",
    check=True,
    timeout=10,
    env=child_env,
)
arguments = json.loads(result.stdout)
print(arguments)  # ['-c', 'Python 学习', '--count', '2']
assert arguments[1:] == ["Python 学习", "--count", "2"]
# '2' 仍是字符串；参数的解释与类型转换由程序负责。

['-c', 'Python 学习', '--count', '2']


## 2 用 argparse 声明输入

### 2.1 位置参数、选项与默认值

ArgumentParser 保存参数规则，add\_argument 声明每项输入，parse\_args 返回带属性的 Namespace。位置参数按位置识别，--count 这样的选项通过名称识别。

| 参数设置 | 中文名称／含义 |
| --- | --- |
| type | 类型转换函数，例如 int 把文本转换为整数 |
| choices | 允许的取值；在类型转换后检查 |
| default | 未提供可选参数时使用的值 |
| help | 帮助信息中的参数说明 |
| action="store\_true" | 出现开关时保存 True，默认 False |

Notebook 显式传入参数列表，避免把内核启动参数当成业务输入。配套脚本则直接调用 parse\_args()，从真正的命令行读取参数。

In [2]:
import argparse

parser = argparse.ArgumentParser(prog="study", description="生成学习标签")
parser.add_argument("title", help="标签文字")
parser.add_argument("--count", type=int, choices=range(1, 4), default=1)
parser.add_argument("--format", choices=("text", "json"), default="text")
parser.add_argument("--verbose", action="store_true", help="显示更多信息")
args = parser.parse_args(["Python", "--count", "2", "--verbose"])
print(vars(args))
# count 为整数 2；未提供 --format 时使用 text；verbose 为 True。
print(vars(parser.parse_args(["SQL"])))
# count 为 1、format 为 text、verbose 为 False。

{'title': 'Python', 'count': 2, 'format': 'text', 'verbose': True}
{'title': 'SQL', 'count': 1, 'format': 'text', 'verbose': False}


### 2.2 帮助与不合法的参数

默认的 -h 或 --help 会打印帮助并正常退出。缺少必需参数、类型转换失败或不符合 choices 时，默认解析器向标准错误输出诊断并以状态 2 退出。

在 Notebook 内直接调用解析器会遇到 SystemExit；下面精确捕获它并核对状态。contextlib 的重定向仅用于收集本单元的输出。

In [3]:
import contextlib
import io

# 帮助与错误都通过 SystemExit 离开解析器；捕获它是为了比较退出状态。
for tokens, expected_status in [
    (["--help"], 0),
    ([], 2),
    (["Python", "--count", "two"], 2),
    (["Python", "--format", "xml"], 2),
]:
    with io.StringIO() as captured:
        with contextlib.redirect_stdout(captured):
            with contextlib.redirect_stderr(captured):
                try:
                    parser.parse_args(tokens)
                except SystemExit as error:
                    assert error.code == expected_status
                else:
                    raise AssertionError("帮助或参数错误没有按预期退出")
        message = captured.getvalue()
    assert "usage:" in message
    print(tokens, "状态", expected_status)  # 四组状态依次为 0、2、2、2。
    print(message.strip())  # 先显示 usage；三种错误分别指出缺少 title、整数转换失败、格式选项无效。
# 帮助退出为 0；三类参数错误均为 2，不能当作成功解析继续处理。

['--help'] 状态 0
usage: study [-h] [--count {1,2,3}] [--format {text,json}] [--verbose] title

生成学习标签

positional arguments:
  title                 标签文字

options:
  -h, --help            show this help message and exit
  --count {1,2,3}
  --format {text,json}
  --verbose             显示更多信息
[] 状态 2
usage: study [-h] [--count {1,2,3}] [--format {text,json}] [--verbose] title
study: error: the following arguments are required: title
['Python', '--count', 'two'] 状态 2
usage: study [-h] [--count {1,2,3}] [--format {text,json}] [--verbose] title
study: error: argument --count: invalid int value: 'two'
['Python', '--format', 'xml'] 状态 2
usage: study [-h] [--count {1,2,3}] [--format {text,json}] [--verbose] title
study: error: argument --format: invalid choice: 'xml' (choose from text, json)


## 3 脚本入口与退出状态

sys.exit 会抛出 SystemExit。主线程中未被捕获时，解释器退出；整数 0 表示成功，非零表示异常结束。具体非零值的含义应由程序说明。

本章脚本约定：参数错误为 2，空白标题这种业务失败为 1，成功和查看帮助为 0。主函数返回整数，入口用 sys.exit(main()) 将返回值传给操作系统。先查看配套脚本，再启动独立进程观察状态。

这里的入口条件和空白标题分支用于演示“主函数返回值如何成为进程状态”，不要求普通示例都增加入口保护或输入检查。

Step 1：在课程目录运行带空格标题，预期两项 JSON 标签和状态 0。

```powershell
python scripts/22-cli-logging-config/study_cli.py "Python 学习" --count 2 --format json
```

配套脚本的实现如下，随后使用子进程观察其状态与输出：

```python
import argparse
import json
import sys


def make_parser() -> argparse.ArgumentParser:
    """声明标签、重复次数和输出格式。"""
    parser = argparse.ArgumentParser(description="生成学习标签")
    parser.add_argument("title", help="标签文字")
    parser.add_argument("--count", type=int, choices=range(1, 4), default=1)
    parser.add_argument("--format", choices=("text", "json"), default="text")
    return parser


def main() -> int:
    """输出标签；空白标题返回 1，成功返回 0。"""
    # 1. argparse 接收真正的命令行参数，语法或取值错误退出为 2。
    args = make_parser().parse_args()
    if not args.title.strip():
        # 空白标题仅向 stderr 输出这条诊断，随后以状态 1 退出。
        print("生成标签失败：title 不能为空白", file=sys.stderr)
        return 1

    # 2. 参数已解析，按输出格式生成最终结果。
    labels = [args.title] * args.count
    if args.format == "json":
        # 标题“Python 学习”、count=2 时为 ["Python 学习", "Python 学习"]。
        print(json.dumps(labels, ensure_ascii=False))
    else:
        # text 格式每个标签占一行；count=2 时打印两行相同标题。
        print("\n".join(labels))
    return 0


if __name__ == "__main__":
    sys.exit(main())
```

In [4]:
cli_path = Path("scripts/22-cli-logging-config/study_cli.py").resolve()

# 每组输入启动独立进程，分别观察成功、帮助、用法错误和业务错误。
for tokens, expected_status in [
    (["Python 学习", "--count", "2", "--format", "json"], 0),
    (["--help"], 0),
    (["Python", "--count", "0"], 2),
    (["   "], 1),
]:
    result = subprocess.run(
        [sys.executable, "-B", str(cli_path), *tokens],
        shell=False,
        capture_output=True,
        encoding="utf-8",
        check=False,
        timeout=10,
        env=child_env,
    )
    assert result.returncode == expected_status
    print("状态", result.returncode)  # 依次为状态 0、0、2、1。
    print((result.stdout + result.stderr).strip())  # 依次显示两项 JSON 标签、帮助、count 取值错误、空白标题诊断。
# check=False 在这里是为了核对约定的非零状态；每次都明确检查 returncode。

状态 0
["Python 学习", "Python 学习"]


状态 0
usage: study_cli.py [-h] [--count {1,2,3}] [--format {text,json}] title

生成学习标签

positional arguments:
  title                 标签文字

options:
  -h, --help            show this help message and exit
  --count {1,2,3}
  --format {text,json}


状态 2
usage: study_cli.py [-h] [--count {1,2,3}] [--format {text,json}] title
study_cli.py: error: argument --count: invalid choice: '0' (choose from 1, 2, 3)
状态 1
生成标签失败：title 不能为空白


## 4 环境变量属于进程输入

os.environ 是环境变量映射，键和值都是字符串；os.getenv 在名称不存在时返回给定默认值，默认是 None。整数和布尔含义都需要应用程序自行解析，字符串 "0" 不会自动变成整数 0。

subprocess.run 的 env 参数替换子进程的环境映射。先复制现有环境，再修改本例的变量，保留系统运行条件；传给子进程的字典不会修改当前进程的 os.environ。

In [5]:
before_limit = os.environ.get("NOTEBOOK_STUDY_LIMIT")
configured_env = child_env.copy()
configured_env["NOTEBOOK_STUDY_LIMIT"] = "0"
probe = (
    "import os; "
    "value = os.getenv('NOTEBOOK_STUDY_LIMIT'); "
    "print(repr(value), type(value).__name__)"
)
result = subprocess.run(
    [sys.executable, "-B", "-c", probe],
    shell=False,
    capture_output=True,
    encoding="utf-8",
    check=True,
    timeout=10,
    env=configured_env,
)
print(result.stdout.strip())  # '0' str
assert os.environ.get("NOTEBOOK_STUDY_LIMIT") == before_limit
# 当前进程的变量值保持原样，子进程中的 '0' 仍需转换。

'0' str


## 5 配置读取与覆盖顺序

### 5.1 TOML 与 JSON 只负责解析

tomllib.load 从二进制文件读取 TOML，返回字典；json.load 可以从文本流读取 JSON。JSON 根值不一定是对象；本例给定的配置文件均使用对象结构。

文本与数据格式章节讲格式规则，本章关注多个配置来源如何合并。下面约定 limit 表示每批最多处理的记录数，范围为 0 到 100，0 表示本批不处理记录。

In [6]:
import tempfile
import tomllib

with tempfile.TemporaryDirectory() as folder:
    config_dir = Path(folder)
    toml_path = config_dir / "study.toml"
    json_path = config_dir / "study.json"
    toml_path.write_text("limit = 8\n", encoding="utf-8")
    json_path.write_text('{"limit": 8}\n', encoding="utf-8")
    with toml_path.open("rb") as stream:
        toml_config = tomllib.load(stream)
    with json_path.open(encoding="utf-8") as stream:
        json_config = json.load(stream)
    print(toml_config == json_config)  # True：两种文件得到相同配置。
# 临时文件已清理；解析出的字典仍可用于后续覆盖示例。

True


### 5.2 明确默认值、文件、环境变量和命令行的优先级

本章约定优先级从高到低为：显式命令行参数、环境变量、配置文件、默认值。这是本例的应用规则，Python 不会自动建立这条顺序。

命令行的 default=None 表示“用户未提供”，实际业务默认值 10 在合并时使用。不能写 cli\_limit or 10，因为合法值 0 也是假值。各来源按给定的合法整数使用；本节只实现优先级选择，不额外检查类型和范围。

In [7]:
from collections.abc import Mapping


def resolve_limit(
    cli_limit: int | None,
    environment: Mapping[str, str],
    file_config: Mapping[str, int],
) -> int:
    """按本章优先级选取每批记录数上限。"""
    # 1. 仅在来源缺失时回退，保留合法的 0。
    if cli_limit is not None:
        value = cli_limit
    elif "NOTEBOOK_STUDY_LIMIT" in environment:
        value = int(environment["NOTEBOOK_STUDY_LIMIT"])
    else:
        value = file_config.get("limit", 10)

    return value


limit_parser = argparse.ArgumentParser(prog="batch")
limit_parser.add_argument("--limit", type=int, default=None)
no_option = limit_parser.parse_args([]).limit
explicit_zero = limit_parser.parse_args(["--limit", "0"]).limit
print(resolve_limit(no_option, {}, {}))  # 10：默认值。
print(resolve_limit(no_option, {}, toml_config))  # 8：文件覆盖默认值。
print(resolve_limit(no_option, {"NOTEBOOK_STUDY_LIMIT": "3"}, toml_config))
# 3：环境变量覆盖文件。
print(resolve_limit(explicit_zero, configured_env, toml_config))  # 0
assert resolve_limit(0, {"NOTEBOOK_STUDY_LIMIT": "7"}, {}) == 0
# 显式参数优先；低优先级值在本例中被覆盖。

10
8
3
0


### 5.3 解析与转换失败直接暴露

TOML 语法错误发生在文件格式解析阶段；存在的环境变量无法转为整数时，错误来自 int。下面分别观察原始异常，不把失败转换成缺失值再使用默认配置。

In [8]:
# 预期 TOMLDecodeError：数组语法未完成。
tomllib.loads("limit = [")

TOMLDecodeError: Invalid value (at end of document)

In [9]:
# 预期 ValueError：环境变量已提供，但 many 无法转为整数。
resolve_limit(None, {"NOTEBOOK_STUDY_LIMIT": "many"}, {})

ValueError: invalid literal for int() with base 10: 'many'

## 6 日志的级别、去向与格式

### 6.1 Logger、Handler 与 Formatter

Logger 接收事件并按级别筛选，Handler 决定发往哪里并可再次筛选，Formatter 决定输出文本。print 适合 CLI 的最终结果，logging 用来描述运行事件。

| 日志级别 | 中文名称／含义 |
| --- | --- |
| DEBUG | 调试细节 |
| INFO | 正常运行信息 |
| WARNING | 出现值得注意的问题，程序仍可继续 |
| ERROR | 某项操作未能完成 |
| CRITICAL | 严重错误，程序可能无法继续 |

级别从 DEBUG 到 CRITICAL 逐渐升高。下面的 Logger 允许 DEBUG，Handler 只接收 WARNING 及以上；Formatter 中的 levelname、name、message 分别表示级别名、记录器名和消息正文。消息参数单独传入，交给 logging 格式化。

In [10]:
import logging

logger = logging.getLogger("notebook22.levels")
previous = (logger.level, logger.propagate, logger.disabled)
with io.StringIO() as log_text:
    handler = logging.StreamHandler(log_text)
    handler.setLevel(logging.WARNING)
    formatter = logging.Formatter("%(levelname)s %(name)s: %(message)s")
    handler.setFormatter(formatter)
    # 先设 Logger 的入口级别，再由 Handler 的级别筛掉低级别记录。
    try:
        logger.setLevel(logging.DEBUG)
        logger.disabled = False
        logger.propagate = False
        logger.addHandler(handler)
        logger.debug("读取候选记录")
        logger.info("读取 %s 条记录", 8)
        logger.warning("跳过 %s 条空记录", 1)
        print(log_text.getvalue().strip())
        # WARNING notebook22.levels: 跳过 1 条空记录
        assert len(log_text.getvalue().splitlines()) == 1
    finally:
        logger.removeHandler(handler)
        handler.close()
        logger.setLevel(previous[0])
        logger.propagate = previous[1]
        logger.disabled = previous[2]
# 只移除自己添加的处理器，并恢复修改过的状态，可重复运行。

WARNING notebook22.levels: 跳过 1 条空记录


### 6.2 传播为什么会造成重复日志

名称中的点建立父子关系，例如 notebook22.tree.worker 的父记录器是 notebook22.tree。propagate 默认 True，记录还会交给祖先记录器的 Handler；传播时不再检查祖先 Logger 的级别，但会检查祖先 Handler 的级别。

同一条记录同时被子、父两处 Handler 输出，就会重复。常用做法是只在合适的祖先处配置 Handler；若某个子记录器必须独立处理，则关闭它的传播。下面先观察重复，再在同一父处理器上消除重复。

In [11]:
parent_logger = logging.getLogger("notebook22.tree")
worker_logger = logging.getLogger("notebook22.tree.worker")
loggers = (parent_logger, worker_logger)
saved = [(item.level, item.propagate, item.disabled) for item in loggers]
with io.StringIO() as log_text:
    parent_handler = logging.StreamHandler(log_text)
    worker_handler = logging.StreamHandler(log_text)
    try:
        parent_logger.setLevel(logging.CRITICAL)
        worker_logger.setLevel(logging.INFO)
        for item in loggers:
            item.disabled = False
        parent_logger.propagate = False
        worker_logger.propagate = True
        parent_logger.addHandler(parent_handler)
        worker_logger.addHandler(worker_handler)
        # 同一记录先由子处理器输出，再沿传播链交给父处理器，因而出现两次。
        worker_logger.info("一次事件")
        assert log_text.getvalue().splitlines() == ["一次事件", "一次事件"]
        print("修正前", log_text.getvalue().splitlines())  # 修正前 ['一次事件', '一次事件']。

        # 移除子处理器后，只保留传播到父处理器的一条输出路径。
        worker_logger.removeHandler(worker_handler)
        log_text.seek(0)
        log_text.truncate(0)
        worker_logger.info("一次事件")
        print("修正后", log_text.getvalue().splitlines())  # 修正后 ['一次事件']。
        assert log_text.getvalue().splitlines() == ["一次事件"]
        # 父 Logger 虽为 CRITICAL，传播仍到其默认 NOTSET 的 Handler。
    finally:
        parent_logger.removeHandler(parent_handler)
        worker_logger.removeHandler(worker_handler)
        parent_handler.close()
        worker_handler.close()
        for item, state in zip(loggers, saved, strict=True):
            item.setLevel(state[0])
            item.propagate = state[1]
            item.disabled = state[2]

修正前 ['一次事件', '一次事件']
修正后 ['一次事件']


### 6.3 重复执行时先辨认配置归属

getLogger 对同名记录器返回同一个对象；重复运行单元后，之前添加的 Handler 仍可能存在。每次新建并添加 Handler 而不移除，会不断增加输出次数。

basicConfig 配置根记录器，根记录器已有 Handler 时默认不做任何事。Notebook 内核可能已经配置日志；前两例使用独立名称并清理自己的 Handler，避免覆盖宿主配置。

In [12]:
first_logger = logging.getLogger("notebook22.levels")
second_logger = logging.getLogger("notebook22.levels")
print(first_logger is second_logger)  # True：同名对象不会因重取而重置。
print(len(first_logger.handlers))  # 0：前面的单元移除了自己添加的 Handler。
assert first_logger.handlers == []
assert parent_logger.handlers == []
assert worker_logger.handlers == []

True
0


## 7 子进程的参数、错误与超时

### 7.1 列表传参并检查完成状态

subprocess.run 等待进程结束，返回 CompletedProcess。shell=False 是默认值；本例显式写出，并直接启动 Python 可执行文件。空格和 & 等字符作为普通参数传递，不作为 shell 命令解释。

capture\_output 捕获标准输出和标准错误，encoding="utf-8" 明确解码方式。check=True 遇到非零状态会抛出 CalledProcessError，其中保留 returncode 以及已捕获的输出。这里的 shell 行为不能直接推广到 Windows 的 .bat 或 .cmd 文件。

In [13]:
literal_title = "Python 学习 & SQL"
result = subprocess.run(
    [sys.executable, "-B", str(cli_path), literal_title, "--format", "json"],
    shell=False,
    capture_output=True,
    encoding="utf-8",
    check=True,
    timeout=10,
    env=child_env,
)
assert json.loads(result.stdout) == [literal_title]
print(result.stdout.strip())  # ["Python 学习 & SQL"]；标题保持为一项。

# 这次故意让子进程失败，观察 CalledProcessError 携带的状态和诊断。
try:
    subprocess.run(
        [sys.executable, "-B", str(cli_path), " "],
        shell=False,
        capture_output=True,
        encoding="utf-8",
        check=True,
        timeout=10,
        env=child_env,
    )
except subprocess.CalledProcessError as error:
    assert error.returncode == 1
    assert "title 不能为空白" in error.stderr
    print(error.returncode, error.stderr.strip())  # 1 生成标签失败：title 不能为空白。
else:
    raise AssertionError("业务失败没有触发 CalledProcessError")

["Python 学习 & SQL"]


1 生成标签失败：title 不能为空白


### 7.2 给等待设置上限

timeout 以秒为单位。run 超时后会终止并等待直接启动的子进程结束，再抛出 TimeoutExpired；这与手动使用 Popen 时的清理责任不同。

超时不是精确的总墙钟时间保证，因为某些平台的进程创建过程无法立即中断。下面只让一个本地子进程短暂休眠，不启动其他进程。

In [14]:
try:
    subprocess.run(
        [sys.executable, "-B", "-c", "import time; time.sleep(5)"],
        shell=False,
        capture_output=True,
        encoding="utf-8",
        check=True,
        timeout=0.2,
        env=child_env,
    )
except subprocess.TimeoutExpired as error:
    assert error.timeout == 0.2
    print(type(error).__name__)  # TimeoutExpired；run 已终止并等待该子进程。
else:
    raise AssertionError("休眠进程没有按预期超时")

TimeoutExpired


## 本章小结

（1）sys.argv 保留字符串参数；argparse 根据声明完成转换、取值检查、帮助和参数错误处理。

（2）退出状态、标准输出和标准错误承担不同职责。调用方应检查状态，不能只看是否有输出。

（3）配置来源的优先级由应用定义。缺失、合法的 0 和无效输入必须分开处理。

（4）日志先经过 Logger，再交给 Handler；传播到祖先时要防止重复处理，重复执行后应恢复自己修改的配置。

（5）子进程用列表传参，明确编码、状态检查和超时，并由调用方式承担相应清理责任。

自查：能否指出一个配置值来自哪一层、一条日志由哪个 Handler 输出，以及一个非零退出状态由谁检查？

## 练习

（1）先预测下面三次配置解析的结果，再运行核对。分别说明来源缺失、显式零值和覆盖的作用；核对标准为预测与输出一致，并能指出每次最终使用的来源。

In [15]:
print(resolve_limit(None, {}, {"limit": 12}))
print(resolve_limit(0, {"NOTEBOOK_STUDY_LIMIT": "7"}, {"limit": 12}))
print(resolve_limit(None, {"NOTEBOOK_STUDY_LIMIT": "7"}, {"limit": 12}))
# 运行前记录预测，运行后按本章优先级逐项解释。

12
0
7


（2）用 argparse 声明 --limit，default=None、type=int；把解析结果交给 resolve_limit。无选项且文件为 4 时应得到 4，显式 --limit 0 时应得到 0；再加入环境变量值 "7"，分别解释省略选项与显式零值的结果。

单独检查 --limit many：int 转换失败由 argparse 报告，并以 SystemExit(2) 退出。精确捕获只用于检查其退出状态，不作成功回退。

In [16]:
exercise_arguments = [[], ["--limit", "0"]]
# 创建解析器，配合文件值 4 检查输出 4、0。
# 加入环境变量 "7" 后检查输出 7、0，说明优先级。
# 单独核对 --limit many 的 SystemExit.code 为 2。

（3）为独立命名的 Logger 配置一个 StringIO Handler。Logger 设为 INFO，Handler 设为 ERROR，发送 INFO、WARNING、ERROR 各一条，检查只有 ERROR 出现。再把 Handler 调为 INFO，检查新发送的 INFO 出现一次。用 finally 移除并关闭自己添加的 Handler，恢复 Logger 原来的级别与传播设置；重复执行不能增加输出次数。

In [17]:
exercise_logger_name = "notebook22.exercise"
# 在这里保存状态、配置阈值、捕获输出并执行清理。
# 检查输出行数与消息内容，而不只检查是否产生过任何输出。

### 提示

第一题按“命令行 → 环境变量 → 文件 → 默认值”查找首个已提供的值。第二题不要把合法零值当作缺失。第三题同时考虑 Logger 与 Handler 的阈值，并在 finally 恢复自己的修改。

### 参考解析

第一题依次为 12、0、7，来源分别是文件、命令行、环境变量。

第二题省略选项时，None 让后续来源参与选择；文件值为 4 且没有环境变量时得到 4，添加环境变量后得到 7。显式 0 始终优先。many 不能转为 int，argparse 将该错误作为参数诊断，以状态 2 退出；不需要另写范围防护。

第三题先保存 Logger 的 level、propagate，设置 INFO 并关闭向父级传播；添加 ERROR 阈值的 StreamHandler 后，前三次发送只留下 ERROR。把同一 Handler 调为 INFO 后，新 INFO 再增加一行。finally 移除并关闭此 Handler、恢复原状态，两轮独立运行各得到两行，不积累重复输出。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| Python 官方文档（3.12） | [sys.argv 的内容](https://docs.python.org/3.12/library/sys.html#sys.argv)、[sys.exit 与退出状态](https://docs.python.org/3.12/library/sys.html#sys.exit)；[位置参数与选项](https://docs.python.org/3.12/library/argparse.html#name-or-flags)、[默认值](https://docs.python.org/3.12/library/argparse.html#default)、[type](https://docs.python.org/3.12/library/argparse.html#type)、[choices](https://docs.python.org/3.12/library/argparse.html#choices)、[开关与帮助动作](https://docs.python.org/3.12/library/argparse.html#action)、[parse\_args](https://docs.python.org/3.12/library/argparse.html#argparse.ArgumentParser.parse_args)、[参数错误状态](https://docs.python.org/3.12/library/argparse.html#argparse.ArgumentParser.error)；[环境变量映射](https://docs.python.org/3.12/library/os.html#os.environ)、[getenv](https://docs.python.org/3.12/library/os.html#os.getenv)；[TOML 二进制读取与异常](https://docs.python.org/3.12/library/tomllib.html#tomllib.load)、[TOML 字符串解析](https://docs.python.org/3.12/library/tomllib.html#tomllib.loads)、[JSON 文件读取](https://docs.python.org/3.12/library/json.html#json.load)；[Logger 与消息参数](https://docs.python.org/3.12/howto/logging.html#loggers)、[Handler 与级别](https://docs.python.org/3.12/howto/logging.html#handlers)、[日志级别](https://docs.python.org/3.12/howto/logging.html#logging-levels)、[传播与重复](https://docs.python.org/3.12/library/logging.html#logging.Logger.propagate)、[Formatter](https://docs.python.org/3.12/library/logging.html#logging.Formatter)、[移除 Handler](https://docs.python.org/3.12/library/logging.html#logging.Logger.removeHandler)、[同名 Logger](https://docs.python.org/3.12/library/logging.html#logging.getLogger)、[Handler 关闭](https://docs.python.org/3.12/library/logging.html#logging.Handler.close)、[日志用途](https://docs.python.org/3.12/howto/logging.html#when-to-use-logging)、[basicConfig](https://docs.python.org/3.12/library/logging.html#logging.basicConfig)；[run 的参数、环境和超时清理](https://docs.python.org/3.12/library/subprocess.html#subprocess.run)、[非零状态异常](https://docs.python.org/3.12/library/subprocess.html#subprocess.CalledProcessError)、[超时异常](https://docs.python.org/3.12/library/subprocess.html#subprocess.TimeoutExpired)、[shell 与 Windows 批处理边界](https://docs.python.org/3.12/library/subprocess.html#security-considerations)。配置优先级、limit 范围及 CLI 的业务状态 1 是本章示例约定。 |